In [1]:
import requests
import time
import json
import os
from datetime import datetime

In [2]:
headers = {"User Agent": "TrendPulse/1.0"}

In [3]:
CATEGORIES = {
    
  "by": "burnt-resistor",
  "descendants": 174,
  "id": 47719740,
  "kids": [47721597, 47720716, 47722508, 47721829, 47723984, 47720357, 47720803, 47720380, 47741555, 47720652, 47722421, 47721279, 47723341, 47748644, 47722980, 47720319, 47720679, 47722174, 47725801, 47733963, 47726238, 47728050, 47723037, 47721552, 47723056, 47720764, 47725331, 47727243, 47721738, 47726074, 47725655, 47720814, 47723181, 47720461, 47721537, 47738313, 47721610, 47729753, 47720560, 47725610, 47721505, 47723156, 47723290, 47750933, 47725433, 47720946, 47730045, 47723434, 47727938, 47722300, 47730673, 47730672, 47721341, 47725106, 47720958, 47727842, 47722619, 47738483, 47720259, 47723152, 47733103, 47719983, 47725857, 47724734, 47723293, 47723500, 47734876, 47735677, 47723656, 47729853, 47727950, 47722369, 47722377, 47721450, 47720617, 47720604],
  "score": 988,
  "time": 1775835448,
  "title": "1D Chess",
  "type": "story",
  "url": "https://rowan441.github.io/1dchess/chess.html"
}

SyntaxError: unmatched '}' (366028171.py, line 3)

In [ ]:
def get_category(title):
    title = title.lower()
    for category, keywords in CATEGORIES.items():
        for word in keywords:
            if word in title:
                return category
    return None

In [ ]:
def fetch_data():
    url = "https://hacker-news.firebaseio.com/v0/topstories.json"
    ids = requests.get(url, headers=headers).json()[:500]
    
    collected = []
    category_count = {cat: 0 for cat in CATEGORIES}
    
    for story_id in ids:
        try:
            res = requests.get(
                f"https://hacker-news.firebaseio.com/v0/item/{story_id}.json",
                headers=headers
            )
            data = res.json()
            
            if not data or "title" not in data:
                continue
            category = get_category(data["title"])
            if category and category_count[category] < 25:
                story = {
                    "post_id": data.get("id"),
                    "title": data.get("title"),
                    "category": category,
                    "score":data.get("score",0),
                    "num_comments": data.get("descendants",0),
                    "author": data.get("by"),
                    "collected_at": datetime.now().strftime("%y-%m-%d %H:%M:%S")
                }
                collected.append(story)
                category_count[category] += 1
                if all(v >= 25 for v in category_count.values()):
                    break
            except Exception as e:
                print(f"Error fetching {story_id}: {e}")
        return collected
def save_json(data):
    os.makedirs("data",exist_ok = True)
    filename = f"data/trends_{datetime.now().strftime('%Y%m%d')}.json"
    
    with open(filename, "w")as f:
        json.dump(data, f, indent=4)
        
        print(f"collected.{len(data)}.stories.saved.to.{filename}")
